# Teacher capture — Colab kit (2026-10-15, v3)

One session, all value: **behavioral pre-gate (~1 min) -> full capture (515
rows) -> download.** The capture file is
`data/finetune/teacher-capture-2026-10-15.jsonl` — 328 re-captures of the
existing base + 187 rebalanced new forms (v2 generator: deficit cards at max
3 gold-pinned forms each, 30 bare topics, 16 order flips, 32 causal rows, 20
fragments, 65 refusals) — built locally by
`artifacts/probes/build-teacher-capture-2026-10-15.py` (396 cited / 119
refusal). Upload it plus `teacher-probe-2026-10-15.jsonl` in cell 5.

**Teacher: Qwen3.8-27B UD-Q4_K_XL (~16 GB).** Chosen over the 2-bit Saluki
after the 2026-10-15 external review: the contract is verbatim
character-for-character quoting, and sub-3-bit quants degrade exact copying
first. An A100 40GB fits Q4 with room for the KV cache; an L4 also works.
Weights cache on Google Drive between sessions.

Cells: 1 env, 2 llama.cpp b11617, 3 weights, 4 serve (4 parallel slots),
5 upload + behavioral pre-gate (9 probe items + 5 capture rows), 6 capture
(4 workers, incremental, resume-by-id, max 250 tokens), 7 download + Drive
copy. Capture runtime is an estimate from 515 rows and ~50-120 output tokens
per row: well under an hour on A100, longer on L4. If the pre-gate fails,
STOP — do not run the capture on a broken setup.

Build-time only: nothing here touches the product app (the engine is
local-only; the shipped student stays a stock GGUF).


In [ ]:
BUILD = "b11617"       # llama.cpp nightly pinned 2026-10-14 (asset names verified)
TEACHER = "qwen38-27b" # default: full-fidelity Q4 (verbatim quotes matter)
                       # "saluki" (2-bit, 7.9 GB) is kept as a fallback only
USE_MTP = False        # parent model only: MTP head recipe (sudoingX/qwen38-mtp).
                       # Try it after the pre-gate; if the server fails to
                       # start, set back to False and re-run cell 4.
WORKERS = 1 if USE_MTP else 4   # MTP serves --parallel 1; the capture cell reads this

import os, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
os.makedirs("/content/llama", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout)


In [ ]:
# llama.cpp CUDA build + its cudart bundle (pinned nightly).
import subprocess
base = f"https://github.com/ggml-org/llama.cpp/releases/download/{BUILD}"
files = [f"llama-{BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz",
         f"cudart-llama-{BUILD}-bin-ubuntu-cuda-12.8-x64.tar.gz"]
for f in files:
    subprocess.run(["curl", "-sSL", "--retry", "3", "-o", f"/content/{f}", f"{base}/{f}"], check=True)
    subprocess.run(["tar", "-xzf", f"/content/{f}", "-C", "/content/llama", "--strip-components=1"], check=True)
print(subprocess.run(["/content/llama/llama-server", "--version"], capture_output=True, text=True).stdout[-300:])


In [ ]:
# Teacher weights (the only big download; Apache-2.0 sources).
# Filenames + repos verified against the Hugging Face file API on 2026-10-15.
# Drive cache: if you allow the mount, weights are fetched once and reused by
# later sessions; otherwise they download fresh into /content/models.
import os, subprocess
FN = "Underdog-Saluki-27B-1.0-IQ2-mix.gguf" if TEACHER == "saluki" else "Qwen3.8-27B-UD-Q4_K_XL.gguf"
REPO = "ConwayResearch/Underdog-Saluki-27B-1.0" if TEACHER == "saluki" else "unsloth/Qwen3.8-27B-GGUF"
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CACHE = "/content/drive/MyDrive/teacher-cache"
except Exception as e:
    print("no Drive cache, plain download:", type(e).__name__)
    CACHE = "/content/models"
os.makedirs(CACHE, exist_ok=True)
if not os.path.isfile(f"{CACHE}/{FN}"):
    !pip install -q huggingface_hub
    !hf download {REPO} {FN} --local-dir {CACHE}
else:
    print("cache hit:", f"{CACHE}/{FN}")
if CACHE != "/content/models" and not os.path.isfile(f"/content/models/{FN}"):
    os.symlink(f"{CACHE}/{FN}", f"/content/models/{FN}")
print(subprocess.run(["ls", "-lh", "/content/models"], capture_output=True, text=True).stdout)


In [ ]:
# Colab quirks, measured 2026-10-15: port 8080 is already bound in the VM
# (bind error / 404s), hence 8000; and the CUDA plugin needs both
# /content/llama and /usr/lib64-nvidia on LD_LIBRARY_PATH (see env below).
# Serve. --jinja enables the model's own chat template.
# --parallel WORKERS: -c is the TOTAL context, split across slots
# (16384 / 4 = 4096 per slot; our prompts are ~1.2-1.5k tokens in, <=250 out).
# --cache-reuse 256 reuses the shared system-prompt prefix between requests.
import os, subprocess, time, urllib.request
env = dict(os.environ,
       LD_LIBRARY_PATH="/content/llama:/usr/lib64-nvidia")
       # ^ /usr/lib64-nvidia holds libcuda.so.1 (the NVIDIA driver
       #   lib) in this Colab image; without it the CUDA plugin fails
       #   to dlopen and llama.cpp silently runs on CPU (measured
       #   2026-10-15: 1.9 tok/s, GPU RAM 0.0/40 GB).
cmd = ["/content/llama/llama-server", "-m", f"/content/models/{FN}",
       "--jinja", "-ngl", "99", "-fa", "on", "-c", "16384",
       "--cache-reuse", "256", "--parallel", str(WORKERS), "--port", "8000"]
if USE_MTP and TEACHER == "qwen38-27b":
    cmd += ["--spec-type", "draft-mtp", "--spec-draft-n-max", "2"]
log = open("/content/llama-server.log", "w")
proc = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2) as r:
            print("health:", r.read().decode()[:200]); break
    except Exception:
        time.sleep(5)
else:
    print("SERVER DID NOT COME UP — if USE_MTP is True, set it False and re-run; log tail:")
    print(open("/content/llama-server.log").read()[-2000:])


In [ ]:
# UPLOAD both files when the picker opens:
#   teacher-probe-2026-10-15.jsonl    (9 behavioral items)
#   teacher-capture-2026-10-15.jsonl  (515 rows)
# Then BEHAVIORAL PRE-GATE (~1 min). Hard stops: thinking leaks, empty
# replies, cap-hit loops, broken format controls. The 7B ceiling items are
# printed, not gated — a 27B that still fails them is still a valid teacher,
# and the result is a measurement for the repo.
import json, statistics, urllib.request
from google.colab import files as _files
_files.upload()

def chat(messages, max_tokens=250):
    payload = {"messages": messages, "temperature": 0, "max_tokens": max_tokens,
               "chat_template_kwargs": {"enable_thinking": False}}
    req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions",
                                 data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=900) as r:
        d = json.loads(r.read())
    m = d["choices"][0]["message"]
    return (m.get("content") or "", m.get("reasoning_content") or "",
            d["choices"][0].get("finish_reason"))

probe = [json.loads(l) for l in open("/content/teacher-probe-2026-10-15.jsonl", encoding="utf-8")]
all_rows = [json.loads(l) for l in open("/content/teacher-capture-2026-10-15.jsonl", encoding="utf-8")]
smoke = [{"id": p["id"], "q": p["q"], "system": p["system"], "user": p["user"]}
         for p in probe]
smoke += [{"id": r["id"], "q": r["q"], "system": r["system"], "user": r["user"]}
          for r in all_rows[:5]]
print(len(smoke), "smoke rows:", len(probe), "probe +", len(smoke) - len(probe), "capture")

answers, hard_fail, est_tokens = {}, [], []
for row in smoke:
    content, reasoning, finish = chat(
        [{"role": "system", "content": row["system"]},
         {"role": "user", "content": row["user"]}])
    answers[row["id"]] = content
    est_tokens.append(len(content) // 4)
    if reasoning.strip():
        hard_fail.append((row["id"], "thinking leak"))
    if not content.strip():
        hard_fail.append((row["id"], "empty reply"))
    if finish == "length":
        hard_fail.append((row["id"], "hit the 250-token cap (loop/verbosity)"))
    label = (row.get("q") or row["id"])[:38]
    print(f"{row['id']:14} q={label!r:40} finish={finish} "
          f"~{len(content)//4:3d}tok {repr(content[:100])}")

if answers.get("france", "").strip() != "Not in this book.":
    hard_fail.append(("france", "out-of-book control not the exact refusal"))
if "[Ch." not in answers.get("watermark", ""):
    hard_fail.append(("watermark", "in-book control carries no [Ch. tag"))
for cid in ("why-does", "whys-kafka", "why-is-kafka", "what-about", "how-do", "auto-scanning"):
    text = answers.get(cid, "")
    state = ("refused" if text.strip().startswith("Not in this book")
             else "cited" if "[Ch." in text else "uncited prose")
    print(f"behavior {cid}: {state}")
print("[Ch. first-token count:", sum(1 for r in smoke if answers.get(r["id"], "").lstrip().startswith("[Ch.")),
      "of", len(smoke))
print("median ~tokens:", statistics.median(est_tokens))
print("PRE-GATE:", "FAIL " + repr(hard_fail[:6]) if hard_fail
      else "PASS — proceed to the capture cell")


In [ ]:
# FULL CAPTURE — every row through the frozen seam prompts, WORKERS at a time.
# Appends after EACH row; resume is keyed by row id (a torn trailing line from
# a killed session is dropped, not misread). max_tokens 250: the contract
# answer is one quote + one gloss line (well under 200 tokens), so the cap
# only catches loops — finish_reason "length" here is a loop signal.
import json, os, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed

rows = [json.loads(l) for l in open("/content/teacher-capture-2026-10-15.jsonl", encoding="utf-8") if l.strip()]
out = f"/content/teacher-capture-results-{TEACHER}.jsonl"

done = set()
if os.path.isfile(out):
    keep, torn = [], 0
    for line in open(out, encoding="utf-8"):
        try:
            keep.append(json.loads(line))
        except Exception:
            torn += 1
    if torn:
        with open(out, "w", encoding="utf-8") as fh:
            for rec in keep:
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print("dropped", torn, "torn line(s) from a previous run")
    done = {rec["id"] for rec in keep}
    print("resuming:", len(done), "rows already saved")

todo = [row for row in rows if row["id"] not in done]
print(len(todo), "to generate |", len(rows), "total | workers", WORKERS)

lock = threading.Lock()
fh = open(out, "a", encoding="utf-8")

def capture(row):
    msgs = [{"role": "system", "content": row["system"]},
            {"role": "user", "content": row["user"]}]
    t0 = time.time()
    try:
        content, reasoning, finish = chat(msgs, max_tokens=250)
        rec = {"id": row["id"], "q": row["q"], "answer": content,
               "finish": finish, "reasoning_chars": len(reasoning),
               "seconds": round(time.time() - t0, 1)}
    except Exception as e:
        rec = {"id": row["id"], "q": row["q"],
               "error": f"{type(e).__name__}: {e}"}
    with lock:
        fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
        fh.flush()
    return rec

finished = 0
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = [pool.submit(capture, row) for row in todo]
    for fut in as_completed(futures):
        rec = fut.result()
        finished += 1
        if finished % 10 == 0 or finished == len(todo):
            text = rec.get("answer") or rec.get("error", "")
            print(f"{finished}/{len(todo)} — last: {rec['id']} "
                  f"finish={rec.get('finish')} {repr(text[:100])}", flush=True)
fh.close()
print("wrote", out)


In [ ]:
# Download the results — copy to Drive first (belt and braces), then download.
# This file comes back to the repo for local build + verification.
import os, shutil
if os.path.isdir("/content/drive/MyDrive/teacher-cache"):
    shutil.copy(out, "/content/drive/MyDrive/teacher-cache/")
    print("copied to Drive:", f"/content/drive/MyDrive/teacher-cache/{os.path.basename(out)}")
from google.colab import files
files.download(out)


## What happens after this session

Back in the repo (local, free): the results file feeds the standard build
step — parse → quote-verbatim checks → refusal checks → seal check →
contamination probe → train jsonl. Added checks from the 2026-10-15 review:

- **class-flip diff** against the old (7B) targets: a stricter 27B may refuse
  rows the 7B answered; every flip is inspected, never silently dropped;
- **gloss length filter**: quote 1-2 sentences, gloss one line (verbose
  targets are trimmed, not trained);
- **`[Ch.` placement census**: answers should start at `[Ch.` (the engine's
  retry prefill position); a teacher that puts tags after prose is
  out-of-distribution for the retry;
- **loop check**: word-5-gram repeats (the probe's `repeat5` metric) in
  addition to cap-stops;
- **training mix**: refusal share targeted at ~16-18% (this capture is 23.1%;
  extra refusal rows are downsampled at build time, not deleted from evidence);
- **epochs**: hold total presentations near x2d's 328×6 = 1968 — the training
  notebook derives epochs from the final row count.

Only then does a training session run. Captured-but-rejected rows stay in the
results file as measured evidence; nothing is trained unseen.
